# FINN → region emissions — R walkthrough

This notebook runs the pipeline using `R/utils_finn.R` for the region
defined in `config/config.yml` (**default: Santa Catarina**; point
`region$boundaries_shp` at any shapefile to change it). It needs the
**IRkernel** (`install.packages('IRkernel'); IRkernel::installspec()`).
Run from the project root. See `README.md`, `docs/methodology.md`,
`docs/data_dictionary.md` and the verified `REFERENCES.md`.

> **Units:** raw FINN gases are mol/day, aerosols kg/day; after step 01
> everything is in **metric tonnes**.

## ▶️ Setup — Google Colab (run these first; they auto-skip when you run locally)

**On Google Colab, one manual step first:** open **Runtime → Change runtime
type**, set it to **R**, and Save. Then upload **FINN_SC.zip** with the
**Files** panel (left sidebar → folder icon → upload). Now run every cell from
the top: the next two cells unzip the project, install the system libraries `sf`
needs, and install the R packages.

**Locally (RStudio / Jupyter+IRkernel):** these cells detect the project is
already on disk and only install any missing R packages.

In [ ]:
# --- Locate / unzip the project (Colab-aware, harmless locally) --------------
IN_COLAB <- nzchar(Sys.getenv('COLAB_RELEASE_TAG')) || dir.exists('/content')
has_project <- function(p='.') file.exists(file.path(p, 'config', 'config.yml'))

# If opened from inside notebooks/, step up to the project root.
if (basename(getwd()) == 'notebooks' && has_project('..')) setwd('..')

# On Colab: if you uploaded the zip via the Files panel, unzip it here.
if (!has_project() && !has_project('FINN_SC')) {
  zips <- list.files(pattern = 'FINN_SC.*\\.zip$')
  if (length(zips)) { unzip(zips[1]); message('unzipped ', zips[1]) }
}
if (!has_project() && has_project('FINN_SC')) setwd('FINN_SC')
message('Working dir: ', getwd(), ' | project found: ', has_project())

In [ ]:
# System libraries for `sf` (Colab only) + the R packages (installs only what
# is missing, so it is safe to re-run and safe locally).
if (IN_COLAB) {
  system('apt-get -qq update')
  system('apt-get -qq install -y libgdal-dev libgeos-dev libproj-dev libudunits2-dev')
}
pkgs <- c('yaml','httr','fs','readr','data.table','dplyr','stringr',
          'purrr','tibble','sf','ggplot2','scales','patchwork')
need <- pkgs[!(pkgs %in% rownames(installed.packages()))]
if (length(need)) install.packages(need, repos = 'https://cloud.r-project.org')
message('R packages ready')

## 0. Setup — source helpers and load the config

In [ ]:
# If the notebook launched inside notebooks/, step up to the project root.
if (basename(getwd()) == 'notebooks') setwd('..')
source('R/utils_finn.R')
cfg <- finn_load_config()
sel <- cfg$selection
bounds <- finn_date_bounds(cfg)      # NULL unless start_date/end_date set
years <- selected_years(cfg)
cat('years:', paste(years, collapse=','), '| date range:',
    if (is.null(bounds)) 'none' else paste(bounds$start, bounds$end), '\n')

## 1. Download FINN (step 00)

Set `RUN_DOWNLOAD <- FALSE` to reuse an existing file.

In [ ]:
RUN_DOWNLOAD <- TRUE
year <- years[[1]]
if (RUN_DOWNLOAD) {
  all_urls <- build_finn_urls()
  urls <- filter_finn_urls(all_urls, version = sel$version,
                           sensor = sel$sensor, year = year)
  if (!is.null(bounds)) urls <- filter_finn_by_date(urls, bounds$start, bounds$end)
  download_finn_files(urls, dest_folder = cfg$paths$raw_finn,
    combined_filename = sprintf('%d_finn_%s_%s.csv', year, sel$version, sel$sensor),
    include_source_url = cfg$download$include_source_url,
    min_file_size_kb = cfg$download$min_file_size_kb,
    timeout_seconds = cfg$download$timeout_seconds)
}
raw <- data.table::fread(file.path(cfg$paths$raw_finn,
        sprintf('%d_finn_%s_%s.csv', year, sel$version, sel$sensor)))
head(raw)

## 2. Convert units to tonnes (step 01)

In [ ]:
converted <- convert_finn_units(raw, cfg)
if (!is.null(bounds)) {   # keep only rows within the exact date range (adds DATE col)
  converted <- filter_rows_by_date(converted, year, bounds$start, bounds$end)
  cat('rows in', bounds$start, '..', bounds$end, '->', nrow(converted), '\n')
}
fs::dir_create(cfg$paths$interim)
data.table::fwrite(converted, file.path(cfg$paths$interim,
        sprintf('%d_finn_%s_%s_converted.csv', year, sel$version, sel$sensor)))
summary(converted[, intersect(c('CO2','CO','PM25','OC'), names(converted)), with = FALSE])

## 3. Clip to the region polygons (step 02: bbox pre-filter + point-in-polygon; 2026-10-03 review)

The bbox is auto-derived from the region shapefile (`bbox: auto`), so we
load the shapefile now and pass it to the clip. Works for any region.

In [ ]:
library(sf)
shp_path <- cfg$region$boundaries_shp
shp <- if (file.exists(shp_path)) st_read(shp_path, quiet = TRUE) else NULL
clipped <- clip_to_region(converted, cfg, shp)   # 2026-10-03 review: bbox + polygons
data.table::fwrite(clipped, file.path(cfg$paths$interim,
  sprintf('filtered_points_%d_%s_%s_converted_within_grid.csv', year, sel$sensor, sel$version)))
cat('records kept inside the region polygons:', nrow(clipped), '\n')
head(clipped)

## 4. Aggregate by region polygon (step 04)

Groups by `region$region_field` (default `NM_MUN`; any polygon layer works).
Needs the boundaries shapefile in `data/inputs/` (see `data/inputs/README.md`).

In [ ]:
stopifnot(!is.null(shp))
region_field <- cfg$region$region_field
summary <- aggregate_by_region(clipped, shp, cfg,
             year = year, sensor = sel$sensor, version = sel$version)
final_sf <- dplyr::left_join(shp, summary, by = region_field)
fs::dir_create(cfg$paths$output)
data.table::fwrite(sf::st_drop_geometry(final_sf),
                   file.path(cfg$paths$output, 'aggregated_emissions_all.csv'))
head(dplyr::arrange(summary, dplyr::desc(events)), 10)  # 2026-10-03 review

## 5. Descriptive analysis (step 05)

For the full figures, run `Rscript R/05_descriptive_analysis.R`. A quick
annual CO / PM2.5 look:

In [ ]:
library(ggplot2); library(dplyr)
agg <- data.table::fread(file.path(cfg$paths$output, 'aggregated_emissions_all.csv'))
agg |>
  filter(version == sel$version, sensor == sel$sensor) |>
  group_by(year) |>
  summarise(CO = sum(CO, na.rm = TRUE), PM25 = sum(PM25, na.rm = TRUE), .groups='drop')

---
Done. Outputs are in `data/output/`. Style
`aggregated_emissions_all.shp` in QGIS (see the QGIS section of `TUTORIAL_FINN_SC.pdf`).